# 02 · Machine Learning

1. **Self-Made Classifier** — XGBoost + Optuna HPO, predicts `selfMade`
2. **Net-Worth Regressor** — XGBoost + Optuna HPO, predicts `log_worth`
3. **SHAP Explainability** — TreeExplainer for both models
4. **Wealth Segment Clustering** — K-Means via `BillionaireClusterer`

All reusable logic lives in `src/billionaires/`.  Notebooks import and call; they never define.

## 0 · Imports & Setup

In [2]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import plotly.express as px

from billionaires.data     import load_raw, validate_schema, clean
from billionaires.features import (
    build_features, get_clf_features, get_reg_features, get_cluster_features,
)
from billionaires.models   import (
    SelfMadeClassifier, WorthRegressor, BillionaireClusterer,
    evaluate_classifier, evaluate_regressor, evaluate_clusters, metrics_table,
)
from billionaires.viz      import confusion_matrix_plot, actual_vs_predicted, shap_summary

FIGURES_DIR = Path("../reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH   = Path("../data/raw/Billionaires Statistics Dataset.csv")
SEED = 42
print("✅ Ready")

✅ Ready


In [3]:
df = build_features(clean(load_raw(DATA_PATH)))
print(f"Shape: {df.shape}")

Shape: (2640, 43)


## 1 · Self-Made Classifier

Predicts whether a billionaire is self-made (`1`) or inherited (`0`) from age, log-worth, and geographic / industry features.

In [4]:
clf_feats = get_clf_features()
# Keep only features that actually exist after engineering
clf_feats = [f for f in clf_feats if f in df.columns]

clf_df = df[clf_feats + ["selfMade"]].dropna()
X_clf  = clf_df[clf_feats].values
y_clf  = clf_df["selfMade"].values

# ── 70 / 15 / 15 split (train / val / test) ──────────────────────────
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X_clf, y_clf, test_size=0.30, random_state=SEED, stratify=y_clf
)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp
)
print(f"Train {X_tr.shape}  |  Val {X_val.shape}  |  Test {X_te.shape}")
print(f"Class balance — 0:{(y_tr==0).sum()}  1:{(y_tr==1).sum()}")

Train (1848, 7)  |  Val (396, 7)  |  Test (396, 7)
Class balance — 0:580  1:1268


In [ ]:
# ── Tune + fit ────────────────────────────────────────────────────────────
clf = SelfMadeClassifier(n_trials=40, seed=SEED)
clf.tune(X_tr, y_tr)
clf.fit(X_tr, y_tr, X_val=X_val, y_val=y_val)

  0%|          | 0/40 [00:00<?, ?it/s]

In [ ]:
clf_metrics = evaluate_classifier(
    y_te, clf.predict(X_te), clf.predict_proba(X_te)[:, 1], split="test"
)
display(metrics_table({"test": clf_metrics}))
print(clf_metrics["classification_report"])

In [ ]:
fig = confusion_matrix_plot(
    y_te, clf.predict(X_te),
    save_path=FIGURES_DIR / "05_confusion.png"
)
plt.show()

## 2 · Net-Worth Regressor

In [ ]:
reg_feats = get_reg_features()
reg_feats = [f for f in reg_feats if f in df.columns]

reg_df = df[reg_feats + ["log_worth"]].dropna()
X_reg  = reg_df[reg_feats].values
y_reg  = reg_df["log_worth"].values

# ── 70 / 15 / 15 split (train / val / test) ──────────────────────────
X_tr_r, X_tmp_r, y_tr_r, y_tmp_r = train_test_split(
    X_reg, y_reg, test_size=0.30, random_state=SEED
)
X_val_r, X_te_r, y_val_r, y_te_r = train_test_split(
    X_tmp_r, y_tmp_r, test_size=0.50, random_state=SEED
)
print(f"Train {X_tr_r.shape}  |  Val {X_val_r.shape}  |  Test {X_te_r.shape}")

In [ ]:
reg = WorthRegressor(n_trials=40, seed=SEED)
reg.tune(X_tr_r, y_tr_r)
reg.fit(X_tr_r, y_tr_r, X_val=X_val_r, y_val=y_val_r)

In [ ]:
reg_metrics = evaluate_regressor(
    y_te_r, reg.predict(X_te_r), exponentiated=True, split="test"
)
display(metrics_table({"test": reg_metrics}))

In [ ]:
fig = actual_vs_predicted(
    y_te_r, reg.predict(X_te_r), r2=reg_metrics["r2"],
    save_path=FIGURES_DIR / "06_actual_vs_pred.png"
)
plt.show()

## 3 · SHAP Explainability

In [ ]:
print("SHAP — Classifier (Self-Made prediction)")
shap_summary(clf.model_, X_te, clf_feats,
             plot_type="bar",
             save_path=FIGURES_DIR / "07_shap_clf_bar.png")

In [ ]:
shap_summary(clf.model_, X_te, clf_feats,
             plot_type="dot",
             save_path=FIGURES_DIR / "07_shap_clf_dot.png")

In [ ]:
print("SHAP — Regressor (Net Worth prediction)")
shap_summary(reg.model_, X_te_r, reg_feats,
             plot_type="dot",
             save_path=FIGURES_DIR / "08_shap_reg_dot.png")

## 4 · Wealth Segment Clustering

Uses `BillionaireClusterer` (K-Means + StandardScaler + PCA).
K is auto-selected via maximum silhouette score across k=2..10.

In [ ]:
# ── Fit BillionaireClusterer (replaces inline K-Means) ───────────────────
cluster_feats = get_cluster_features()
cluster_feats = [f for f in cluster_feats if f in df.columns]  # same guard as clf/reg

cluster_df = df[cluster_feats].dropna().copy()
X_c = cluster_df.to_numpy()

# Fit: elbow search across k=2..10, selects k via max silhouette
clusterer = BillionaireClusterer(k_range=(2, 10), seed=SEED)
clusterer.fit(X_c)

# ── Diagnostics: inertia + silhouette per k ────────────────────────────
diag = clusterer.diagnostics()
display(diag)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(diag.index, diag["inertia"],    "bo-", lw=2)
ax1.set_xlabel("k"); ax1.set_ylabel("Inertia"); ax1.set_title("Elbow Method")
ax2.plot(diag.index, diag["silhouette"], "rs-", lw=2)
ax2.set_xlabel("k"); ax2.set_ylabel("Silhouette Score")
ax2.set_title("Silhouette Score  (higher = better)")
plt.tight_layout(); plt.show()

print(f"Auto-selected k = {clusterer.k_}")

In [ ]:
# ── PCA 2D scatter ─────────────────────────────────────────────────────
cluster_df = cluster_df.copy()
cluster_df["cluster"] = clusterer.labels_

pcs = clusterer.transform_pca(X_c)     # scaler already fitted inside clusterer
cluster_df["PC1"] = pcs[:, 0]
cluster_df["PC2"] = pcs[:, 1]

fig = px.scatter(
    cluster_df, x="PC1", y="PC2",
    color=cluster_df["cluster"].astype(str),
    symbol="selfMade",
    hover_data=["log_worth", "age"],
    title=f"K-Means (k={clusterer.k_}) — PCA 2D Projection",
)
fig.update_traces(marker_size=6, opacity=0.7)
fig.show()

# ── Cluster quality metrics ─────────────────────────────────────────────
X_scaled = clusterer.scaler_.transform(X_c)
cluster_quality = evaluate_clusters(X_scaled, clusterer.labels_, split="full")
display(metrics_table({"clusters": cluster_quality}))

# ── Per-cluster mean / median profiles ─────────────────────────────────
print("\nCluster profiles (mean / median per feature):")
display(clusterer.cluster_profiles(cluster_df, cluster_feats))